# Injury Intelligence - EDA

Author: Angela Lekivetz

This notebook performs exploratory analysis of injury narratives to inform classification and RAG pipeline design. OSHA report is sourced from [OSHA Accident and Injury Data](https://www.kaggle.com/datasets/ruqaiyaship/osha-accident-and-injury-data-1517).

In [1]:
import pandas as pd
import regex as re
import spacy

## 1. Load and Inspect Data

This dataset contains abstracts of the accidents and injuries of workers from 2015-2017. There is some structured data around the unstructured text abstracts, such as Degree of Injury, Body Part(s) Affected, and Construction End Use. 

In [2]:
df = pd.read_csv('../data/osha_injuries_raw.csv')
print(df.info())
print(f'\nNulls: \n{df.isnull().sum()}')  
print(f'\nShape of dataframe: {df.shape}')
print(f'\nColumns: {df.columns}\n')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4847 entries, 0 to 4846
Data columns (total 29 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   summary_nr            4847 non-null   int64 
 1   Event Date            4847 non-null   object
 2   Abstract Text         4847 non-null   object
 3   Event Description     4847 non-null   object
 4   Event Keywords        4847 non-null   object
 5   con_end               4847 non-null   object
 6   Construction End Use  4847 non-null   object
 7   build_stor            4847 non-null   int64 
 8   Building Stories      4847 non-null   object
 9   proj_cost             4847 non-null   object
 10  Project Cost          4847 non-null   object
 11  proj_type             4847 non-null   object
 12  Project Type          4847 non-null   object
 13  Degree of Injury      4847 non-null   object
 14  nature_of_inj         4847 non-null   int64 
 15  Nature of Injury      4845 non-null   

In [3]:
# Keep only relevant columns
df = df[['Abstract Text', 'Event type', 'Event Keywords', 'Nature of Injury', 'Part of Body']]

## 2. Target Variable

The target variable is chosen to be event type, as it maps directly to the narrative text and has intuitive classes (fall, struck-by, etc.)

14 heavily imbalanced classes are found in the dataset, so a threshold of 140 was chosen to drop rare or ambiguous events. "Struck against" and "Fall (same level)" were removed as they had insufficient samples to train on and showed high confusion with similar classes ("Struck-by" and "Fall from elevation") in the initial model.

After filtering, 6 classes remained across 4,427 total rows.

In [4]:
df_events = df['Event type'].value_counts() 
classes = df_events[df_events.values > 140].index.tolist()
df_filtered = df[df['Event type'].isin(classes)]
print(df_filtered['Event type'].value_counts())
print(len(df_filtered))

Event type
Fall (from elevation)        1179
Struck-by                    1138
Caught in or between         1133
Other                         643
Shock                         194
Card-vascular/resp. fail.     179
Name: count, dtype: int64
4466


## 3. Text Analysis

Analyzing the abstract text column gives a mean of 72 words with a median of 59, meaning reasonably short narratives. 

Analyzing a few of the raw narratives, we see that the event type is often inferrable from the text, and that all narratives are written in third person and a consistent style. 

There is also a timestamp attached to the beginning of each narrative which will be stripped for this analysis. 

In [5]:
print(df_filtered['Abstract Text'].isnull().sum())
df_filtered = df_filtered.copy()

df_filtered['text_len'] = df_filtered['Abstract Text'].str.split().str.len()
print(df_filtered['text_len'].describe())

0
count    4466.000000
mean       71.987237
std        48.745109
min         1.000000
25%        42.000000
50%        59.000000
75%        86.000000
max       538.000000
Name: text_len, dtype: float64


In [6]:
for event in df_filtered['Event type'].unique():
    sample = df_filtered[df_filtered['Event type'] == event]['Abstract Text'].iloc[0]
    print(f'[{event}]')
    print(sample[:200])
    print()

[Caught in or between]
At 9:00 a.m. on August 10, 2017, an employee was operating a 400 ton Bliss Coin "Knuckle" mechanical power press. The press was actuated while the employee's right hand was in the point of operation. 

[Other]
At 7:30 a.m. on June 30, 2017, an employee was inserting a match fuze into a fireworks charge.  The fireworks exploded and both of the employee's hands were amputated. 

[Fall (from elevation)]
At 2:00 p.m. on June 30, 2017, an employee was installing a self-adhering membrane to a roof with a slope of 1.25 feet to 12 feet (vertical to horizontal). As the employee installed the last section o

[Struck-by]
At 12:20 p.m. on June 23, 2017, an employee was delivering a load of plywood to the project site. The employee cut the plastic straps holding the plywood together and it slid out from the truck bed, s

[Card-vascular/resp. fail.]
On May 30, 2017, an employee came to work on Sunday to clean as is customary. The employee went home with respiratory issues.  He

## 4. Preprocessing

Each narrative is cleaned by stripping the leading timestamp, lowercasing, removing numbers and punctuation, and filtering single character tokens.

For the TF-IDF baseline, lemmatization is also applied, reducing words to their base form (e.g. "operating" to "operate") and removing stop words. This reduces the feature space and helps the vectorizer treat different forms of the same word as one token.

DistilBERT receives the cleaned but non-lemmatized text, as it was pretrained on natural language and handles word forms internally.

In [7]:
def clean_text(text):
    text = ','.join(text.split(',')[2:])
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = ' '.join([w for w in text.split() if len(w) > 1])
    return text

df_filtered = df_filtered.copy()
df_filtered['clean_text'] = df_filtered['Abstract Text'].apply(clean_text)
print(df_filtered['clean_text'].head(10))

0    an employee was operating ton bliss coin knuck...
1    an employee was using battery operated drill t...
2    an employee was inserting match fuze into fire...
3    an employee was installing self adhering membr...
4    an employee was delivering load of plywood to ...
5    an employee was sorting pistachios inside an a...
6    nineteen employees were completing several wor...
7    an employee was operating an oil rig used for ...
8    an employee was applying adhesive to insulated...
9    an employee set up the rolling roof rigger por...
Name: clean_text, dtype: object


In [8]:
nlp = spacy.load('en_core_web_sm')

def lemmatize(text):
    doc = nlp(text)
    return ' '.join([token.lemma_ for token in doc if not token.is_stop])

sample = df_filtered['clean_text'].iloc[0]
print(df_filtered['Abstract Text'].iloc[0])
print('Before:', sample)
print('After:', lemmatize(sample))

At 9:00 a.m. on August 10, 2017, an employee was operating a 400 ton Bliss Coin "Knuckle" mechanical power press. The press was actuated while the employee's right hand was in the point of operation. The employee's right ring and middle fingers were amputated. Coin "Knuckle" mechanical power press. The press was actuated while the employee's right hand was in the point of operation. The employee's right ring and middle fingers were amputated. 
Before: an employee was operating ton bliss coin knuckle mechanical power press the press was actuated while the employee right hand was in the point of operation the employee right ring and middle fingers were amputated coin knuckle mechanical power press the press was actuated while the employee right hand was in the point of operation the employee right ring and middle fingers were amputated
After: employee operate ton bliss coin knuckle mechanical power press press actuate employee right hand point operation employee right ring middle finger 

In [9]:
df_filtered = df_filtered.copy()
df_filtered['lemma_text'] = df_filtered['clean_text'].apply(lemmatize)

# Drop any nulls for classification step
df_filtered = df_filtered.dropna(subset=['lemma_text'])
df_filtered = df_filtered[df_filtered['lemma_text'] != '']

In [10]:
print(df_filtered.isnull().sum())

Abstract Text       0
Event type          0
Event Keywords      0
Nature of Injury    0
Part of Body        0
text_len            0
clean_text          0
lemma_text          0
dtype: int64


## 5. Save Data

Save the filtered data as `osha_filtered.csv` in the `data/` folder for our next steps. 

In [11]:
df_filtered.to_csv('../data/osha_filtered.csv', index=False)